# Image to LaTeX -- Full Pipeline
Run each cell in order. Make sure GPU is enabled:
**Runtime > Change runtime type > T4 GPU**

## 1. Mount Google Drive & Install Dependencies

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install torch torchvision matplotlib nltk Pillow distance -q

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Load Project (fast restore after runtime reset)
**First time only:** run cell 2a to create `Data_Im2Latx.tar.gz` on Drive.
Then always run cell 2b — copies one archive + extracts locally.

In [2]:
# === 2. RUN AFTER EVERY RUNTIME RESET ===

!mkdir -p /content/CV_220704007

# Copy .py files (tiny, instant)
!cp /content/drive/MyDrive/CV_220704007/*.py /content/CV_220704007/

# Copy zip + extract to local SSD (fast — one big file, -o = overwrite without asking)
!cp /content/drive/MyDrive/CV_220704007/Data_Im2Latx.zip /content/
!unzip -qo /content/Data_Im2Latx.zip -d /content/CV_220704007/
!rm /content/Data_Im2Latx.zip

# Restore checkpoints from Drive if they exist (resume training)
import os
drive_ckpt = '/content/drive/MyDrive/CV_220704007-results/checkpoints'
if os.path.exists(drive_ckpt):
    !cp -r {drive_ckpt} /content/CV_220704007/checkpoints
    print('Restored checkpoints from Drive!')

%cd /content/CV_220704007
!ls

Restored checkpoints from Drive!
/content/CV_220704007
bleu_score.py  Data_Im2Latx	 __MACOSX	 predict.py   vocab.pkl
checkpoints    dataset.py	 model.py	 __pycache__  vocab.py
config.py      edit_distance.py  postprocess.py  train.py


## 3. Check GPU

In [3]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


## 4. Check Dataset

In [4]:
import os
data_dir = "Data_Im2Latx"
print("Dataset contents:")
for item in os.listdir(data_dir):
    full = os.path.join(data_dir, item)
    if os.path.isdir(full):
        n = len(os.listdir(full))
        print("  {}/  ({} files)".format(item, n))
    else:
        sz = os.path.getsize(full)
        print("  {}  ({:.1f} KB)".format(item, sz/1024))

Dataset contents:
  images_val/  (8370 files)
  train_formulas.txt  (11233.3 KB)
  images_test/  (10355 files)
  .DS_Store  (6.0 KB)
  images_train/  (75275 files)
  validation_formulas.txt  (1252.2 KB)


## 5. Build Vocabulary

In [5]:
!python vocab.py

Found 75275 formulas in /content/CV_220704007/Data_Im2Latx/train_formulas.txt
Vocab: 544 tokens total
Saved vocab (544 tokens) -> /content/CV_220704007/vocab.pkl
Original : \widetilde \gamma _ { \mathrm { h o p f } } \simeq \sum _
Encoded  : [498, 224, 507, 540, 306, 540, 517, 526, 527, 515, 542, 542, 421, 444, 507]
Decoded  : \widetilde \gamma _ { \mathrm { h o p f } } \simeq \sum _


## 6. Train (with attention)
This takes ~1-2 hours on a T4 GPU.

In [ ]:
!python train.py

# Auto-save checkpoints to Drive after training
import shutil, os
save_dir = '/content/drive/MyDrive/CV_220704007-results/checkpoints'
os.makedirs(save_dir, exist_ok=True)
if os.path.exists('checkpoints'):
    shutil.copytree('checkpoints', save_dir, dirs_exist_ok=True)
    print('Checkpoints saved to Drive!')

Device: cuda
Loaded vocab (544 tokens) <- /content/CV_220704007/vocab.pkl
[FormulaDataset] 75275 samples from /content/CV_220704007/Data_Im2Latx/images_train
[FormulaDataset] 8370 samples from /content/CV_220704007/Data_Im2Latx/images_val
Parameters: 10,052,768
Attention: True
  ep 1 step 50 loss=3.5174 tf=1.00
  ep 1 step 100 loss=3.2221 tf=1.00
  ep 1 step 150 loss=3.0563 tf=1.00
  ep 1 step 200 loss=2.9433 tf=1.00
  ep 1 step 250 loss=2.8562 tf=1.00
  ep 1 step 300 loss=2.7884 tf=1.00
  ep 1 step 350 loss=2.7346 tf=1.00
  ep 1 step 400 loss=2.6885 tf=1.00
  ep 1 step 450 loss=2.6470 tf=1.00
  ep 1 step 500 loss=2.6109 tf=1.00
  ep 1 step 550 loss=2.5827 tf=1.00
  ep 1 step 600 loss=2.5559 tf=1.00
  ep 1 step 650 loss=2.5321 tf=1.00
  ep 1 step 700 loss=2.5094 tf=1.00
  ep 1 step 750 loss=2.4894 tf=1.00
  ep 1 step 800 loss=2.4703 tf=1.00
  ep 1 step 850 loss=2.4520 tf=1.00
  ep 1 step 900 loss=2.4358 tf=1.00
  ep 1 step 950 loss=2.4191 tf=1.00
  ep 1 step 1000 loss=2.4032 tf=1.00
  

## 7. View Training Plots

In [ ]:
from IPython.display import Image, display
print("--- Loss ---")
display(Image("plots/loss.png"))
print("--- BLEU ---")
display(Image("plots/bleu.png"))
print("--- Edit Distance ---")
display(Image("plots/edit_distance.png"))

## 8. Generate Test Predictions (beam search + post-processing)

In [ ]:
!python predict.py --beam --postprocess

## 9. Preview Predictions

In [ ]:
with open("test_formulas.txt", "r") as f:
    lines = f.readlines()
print("Total predictions:", len(lines))
print("\nFirst 10:")
for i, line in enumerate(lines[:10]):
    print("  [{}] {}".format(i, line.strip()))

## 10. Evaluate on Validation Set (optional)
Uses the provided evaluation scripts.

In [ ]:
# first generate val predictions
# (we reuse predict.py but point it at val images -- quick hack)
import config as C
import torch
from torch.utils.data import DataLoader
from vocab import Vocab
from dataset import FormulaDataset, collate_train
from model import Im2Latex

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vocab = Vocab.load()
ckpt = torch.load("checkpoints/model_best.pt", map_location=dev)
model = Im2Latex(ckpt["vocab_size"]).to(dev)
model.load_state_dict(ckpt["model"])
model.eval()

val_ds = FormulaDataset(C.VAL_IMAGES_DIR, C.VAL_FORMULAS, vocab, training=False)
val_ld = DataLoader(val_ds, C.BATCH, shuffle=False, collate_fn=collate_train)

val_preds = []
for imgs, tgts, lens in val_ld:
    imgs = imgs.to(dev)
    seqs = model.greedy(imgs, vocab.sos_id, vocab.eos_id)
    for seq in seqs:
        val_preds.append(" ".join(vocab.decode(seq)))

with open("val_predictions.txt", "w") as f:
    for p in val_preds:
        f.write(p + "\n")
print("Wrote {} val predictions".format(len(val_preds)))

In [ ]:
print("=== BLEU ===")
!python bleu_score.py --target-formulas Data_Im2Latx/validation_formulas.txt --predicted-formulas val_predictions.txt --ngram 4

print("\n=== Edit Distance ===")
!python edit_distance.py --target-formulas Data_Im2Latx/validation_formulas.txt --predicted-formulas val_predictions.txt

## 11. Save Results to Google Drive

In [ ]:
import shutil, os

save_dir = "/content/drive/MyDrive/CV_220704007-results"
os.makedirs(save_dir, exist_ok=True)

shutil.copy("test_formulas.txt", save_dir)
shutil.copytree("plots", os.path.join(save_dir, "plots"), dirs_exist_ok=True)
shutil.copytree("checkpoints", os.path.join(save_dir, "checkpoints"), dirs_exist_ok=True)

print("Saved everything to", save_dir)
print("Contents:")
for root, dirs, files in os.walk(save_dir):
    for f in files:
        print("  ", os.path.join(root, f))

## 12. Download Results Directly

In [ ]:
from google.colab import files
files.download("test_formulas.txt")
files.download("plots/loss.png")
files.download("plots/bleu.png")
files.download("plots/edit_distance.png")

---
**Done!** Copy the BLEU and Edit Distance numbers from cell 10 into `report.md`.